In [123]:
from pathlib import Path
import pandas as pd

processed_root = Path(
    r"C:\Users\Ander\Desktop\F1-project\data\processed"
)

parquet_files = list(processed_root.rglob("*.parquet"))

print(f"Found {len(parquet_files)} parquet files")

Found 2158 parquet files


In [ ]:
from pathlib import Path
import pandas as pd

processed_root = Path(
    r"C:\Users\Ander\Desktop\F1-project\data\processed"
)

records = []

EXCLUDE_FILES = {
    "all_race_laps_metadata_enriched.parquet",
    "all_race_laps_metadata_with_times.parquet",
    "all_race_laps_metadata.parquet",
    "all_race_laps_metadata_enriched.parquet"
}

for file in processed_root.rglob("*.parquet"):

    if file.name in EXCLUDE_FILES:
        continue

    parts = file.parts

    filename = file.stem

    # -----------------------------
    # Parse metadata
    # -----------------------------

    try:
        race = parts[-3]
        session = parts[-2]
    except:
        race = None
        session = None

    # -----------------------------
    # Determine file type
    # -----------------------------

    if filename.endswith("_laps"):

        file_type = "laps"
        driver = filename.replace("_laps", "")

    elif filename in [
        "weather",
        "corners",
        "drivers",
        "rcm",
        "session_laptimes",
    ]:

        file_type = filename
        driver = None

    else:

        file_type = "telemetry"
        driver = filename

    records.append({
        "path": str(file),
        "race": race,
        "session": session,
        "type": file_type,
        "driver": driver,
    })

registry = pd.DataFrame(records)

display(registry.head())

print(registry.shape)

,path,race,session,type,driver
0,C:\Users\Ander\Desktop\F1-project\data\process...,Abu Dhabi Grand Prix,Qualifying,telemetry,ALB
1,C:\Users\Ander\Desktop\F1-project\data\process...,Abu Dhabi Grand Prix,Qualifying,laps,ALB
2,C:\Users\Ander\Desktop\F1-project\data\process...,Abu Dhabi Grand Prix,Qualifying,telemetry,ALO
3,C:\Users\Ander\Desktop\F1-project\data\process...,Abu Dhabi Grand Prix,Qualifying,laps,ALO
4,C:\Users\Ander\Desktop\F1-project\data\process...,Abu Dhabi Grand Prix,Qualifying,telemetry,ANT


(2156, 5)


In [125]:
import sys
from pathlib import Path

# go up from notebooks/ to project root
sys.path.append(str(Path.cwd().parent))

In [128]:
from src.data.load_driver_telemetry import load_driver_session

import numpy as np
import pandas as pd

laps = pd.read_parquet(
    r"C:/Users/Ander/Desktop/F1-project/data/processed/all_race_laps_metadata_with_times.parquet"
)

# collect telemetry-derived lap rows
extra_rows = []

for row in registry.itertuples():

    # only race laps
    if row.type != "laps":
        continue

    if row.session != "Race":
        continue

    print(f"Processing {row.race} | {row.driver}")

    _, laps_data = load_driver_session(
        registry,
        row.race,
        "Race",
        row.driver
    )

    cols_to_keep = [
    "race",
    "driver",
    "lap",
    "compound",
    "stint",
    "life",
    "fresh",
    "wAT",
    "wTT",
    "wH",
    "wP",
    "wR",
    "wWS",
    "drv",
    "team",
    "sesT"
    ]

    extra_rows.append(
        laps_data[cols_to_keep]
    )


# combine all drivers/races
extra_df = pd.concat(extra_rows, ignore_index=True)

# merge onto main laps dataframe
laps = laps.merge(
    extra_df,
    on=["race", "driver", "lap"],
    how="left"
)

display(laps.head())

Processing Abu Dhabi Grand Prix | ALB
Processing Abu Dhabi Grand Prix | ALO
Processing Abu Dhabi Grand Prix | ANT
Processing Abu Dhabi Grand Prix | BEA
Processing Abu Dhabi Grand Prix | BOR
Processing Abu Dhabi Grand Prix | COL
Processing Abu Dhabi Grand Prix | GAS
Processing Abu Dhabi Grand Prix | HAD
Processing Abu Dhabi Grand Prix | HAM
Processing Abu Dhabi Grand Prix | HUL
Processing Abu Dhabi Grand Prix | LAW
Processing Abu Dhabi Grand Prix | LEC
Processing Abu Dhabi Grand Prix | NOR
Processing Abu Dhabi Grand Prix | OCO
Processing Abu Dhabi Grand Prix | PIA
Processing Abu Dhabi Grand Prix | RUS
Processing Abu Dhabi Grand Prix | SAI
Processing Abu Dhabi Grand Prix | STR
Processing Abu Dhabi Grand Prix | TSU
Processing Abu Dhabi Grand Prix | VER
Processing Australian Grand Prix | ALB
Processing Australian Grand Prix | ALO
Processing Australian Grand Prix | ANT
Processing Australian Grand Prix | BEA
Processing Australian Grand Prix | BOR
Processing Australian Grand Prix | DOO
Proces

,idx,race,driver,lap,lap_time,compound,stint,life,fresh,wAT,wTT,wH,wP,wR,wWS,drv,team,sesT
0,0,Abu Dhabi Grand Prix,ALB,1,99.757,SOFT,1,1,True,26.8,31.4,60.0,1016.5,False,2.9,ALB,Williams,3600.491
1,1,Abu Dhabi Grand Prix,ALB,2,91.710,SOFT,1,2,True,26.8,31.4,60.0,1016.5,False,2.3,ALB,Williams,3692.201
2,2,Abu Dhabi Grand Prix,ALB,3,90.224,SOFT,1,3,True,26.8,31.5,60.0,1016.5,False,1.2,ALB,Williams,3782.425
3,3,Abu Dhabi Grand Prix,ALB,4,90.874,SOFT,1,4,True,26.8,31.4,60.0,1016.5,False,2.0,ALB,Williams,3873.299
4,4,Abu Dhabi Grand Prix,ALB,5,90.314,SOFT,1,5,True,26.8,31.3,61.0,1016.5,False,2.9,ALB,Williams,3963.613


In [132]:
# ============================================================
# Fix dtypes before parquet save
# ============================================================

numeric_cols = [
    "lap_time",
    "stint",
    "life",
    "wAT",
    "wTT",
    "wH",
    "wP",
    "wWS",
    "sesT"
]

for col in numeric_cols:
    laps[col] = pd.to_numeric(
        laps[col],
        errors="coerce"
    )

# rainfall + fresh are booleans
bool_cols = [
    "fresh",
    "wR"
]

for col in bool_cols:
    laps[col] = laps[col].astype("boolean")

# strings/categoricals
string_cols = [
    "race",
    "driver",
    "compound",
    "drv",
    "team"
]

for col in string_cols:
    laps[col] = laps[col].astype("string")

# ============================================================
# Save parquet
# ============================================================

laps.to_parquet(
    r"C:/Users/Ander/Desktop/F1-project/data/processed/all_race_laps_metadata_enriched.parquet"
)